# Chapter 5 · Memory Systems for Agents
**Mastering Agentic AI** — Manning Publications



## What this notebook covers
- The four types of agent memory and when to use each
- SemanticMemory: structured user facts persisted to disk
- EpisodicMemory: session summaries with keyword-based retrieval
- InContextMemory: sliding-window conversation management
- RAG pattern: retrieve relevant episodes before each response
- The MemoryEnabledDietCoach combining all three layers

---

## 5.0 · Setup

In [ ]:
# !pip install openai python-dotenv
import os, json, datetime, hashlib, statistics
from pathlib import Path
from typing import Any
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY — see appendix_a_api_keys.md"
client = OpenAI()
MODEL = "gpt-4o-mini"
print("Ready")

## 5.1 · The Four Types of Agent Memory

| Type | What it stores | Lifetime | Production store |
|---|---|---|---|
| **In-context** | Current conversation turns | Session only | Context window (list of dicts) |
| **Episodic** | Summaries of past sessions | Persistent | Vector DB (FAISS, Pinecone) |
| **Semantic** | User facts and preferences | Persistent | Key-value store (Redis) |
| **Procedural** | Skills and workflows | Persistent | SKILL.md files (Chapter 3) |

The key insight: each type solves a different problem. In-context memory is fastest but ephemeral. Episodic and semantic memory persist but require retrieval. Procedural memory (Skills) is injected at agent startup and rarely changes.

> **Andrew Ng:** "Do not reach for a vector database before you have tried a simple JSON file. Retrieval complexity should match your actual data size.

## 5.2 · SemanticMemory — Storing User Facts

In [ ]:
class SemanticMemory:
    """
    Stores structured facts about a user that persist across sessions.
    
    Production upgrade path:
      JSON file (this chapter) → Redis → DynamoDB
    The interface stays identical — only the backend changes.
    """
    def __init__(self, user_id: str, store_path: Path = Path(".memory")):
        self.user_id = user_id
        self.path = store_path / f"{user_id}_semantic.json"
        self.path.parent.mkdir(exist_ok=True)
        self._data: dict = json.loads(self.path.read_text()) if self.path.exists() else {}

    def set(self, key: str, value: Any) -> None:
        self._data[key] = value
        self.path.write_text(json.dumps(self._data, indent=2))

    def get(self, key: str, default: Any = None) -> Any:
        return self._data.get(key, default)

    def all(self) -> dict:
        return dict(self._data)

    def as_context_string(self) -> str:
        """Formats memory as a human-readable context block for the system prompt."""
        if not self._data:
            return "No user profile stored yet."
        lines = [f"  {k}: {v}" for k, v in self._data.items()]
        return "User profile (from semantic memory):\n" + "\n".join(lines)

# Test SemanticMemory
mem = SemanticMemory("alex_demo")
mem.set("name", "Alex")
mem.set("weight_kg", 78)
mem.set("goal", "lose 5kg")
mem.set("restriction", "lactose intolerant")
mem.set("activity", "moderate — 3 gym sessions per week")
print(mem.as_context_string())

## 5.3 · EpisodicMemory — Session Summaries

In [ ]:
class EpisodicMemory:
    """
    Stores summaries of past coaching sessions.
    
    Each episode: {date, summary, key_insights, goal_set}
    
    Production upgrade: embed summaries with sentence-transformers,
    index with FAISS. The retrieve() method becomes a semantic search
    instead of a keyword search.
    """
    def __init__(self, user_id: str, store_path: Path = Path(".memory")):
        self.user_id = user_id
        self.path = store_path / f"{user_id}_episodes.json"
        self.path.parent.mkdir(exist_ok=True)
        self._episodes: list[dict] = json.loads(self.path.read_text()) if self.path.exists() else []

    def add_episode(self, summary: str, key_insights: list[str], goal_set: str | None = None) -> None:
        episode = {
            "date":         datetime.date.today().isoformat(),
            "summary":      summary,
            "key_insights": key_insights,
            "goal_set":     goal_set,
            "id":           hashlib.md5(f"{self.user_id}{datetime.datetime.now()}".encode()).hexdigest()[:8],
        }
        self._episodes.append(episode)
        self.path.write_text(json.dumps(self._episodes, indent=2))
        return episode

    def retrieve(self, query: str, n: int = 3) -> list[dict]:
        """Keyword-based retrieval — replace with vector search in production."""
        query_words = set(query.lower().split())
        scored = []
        for ep in self._episodes:
            text = f"{ep['summary']} {' '.join(ep.get('key_insights', []))}".lower()
            overlap = len(query_words & set(text.split()))
            scored.append((overlap, ep))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [ep for _, ep in scored[:n]]

    def get_recent(self, n: int = 3) -> list[dict]:
        return self._episodes[-n:]

    def as_context_string(self, query: str | None = None, n: int = 3) -> str:
        episodes = self.retrieve(query, n) if query else self.get_recent(n)
        if not episodes:
            return "No past sessions recorded."
        lines = []
        for ep in episodes:
            lines.append(f"  [{ep['date']}] {ep['summary']}")
            if ep.get("goal_set"):
                lines.append(f"    Goal: {ep['goal_set']}")
        return "Relevant past sessions:\n" + "\n".join(lines)

# Simulate past sessions
ep_mem = EpisodicMemory("alex_demo")
ep_mem.add_episode(
    summary="Alex set a 5kg weight loss goal. Identified protein deficit (only 45g/day vs 62g target).",
    key_insights=["low protein intake", "skips breakfast", "high fibre from vegetables"],
    goal_set="Eat 30g of protein at breakfast every day this week."
)
ep_mem.add_episode(
    summary="Alex reported completing the breakfast goal 5/7 days. Energy levels improved.",
    key_insights=["improved adherence", "energy improvement", "still skips breakfast 2 days"],
    goal_set="Add a protein shake on the two days when breakfast is skipped."
)
print(ep_mem.as_context_string(query="protein breakfast"))

## 5.4 · InContextMemory — Sliding Window Management

In [ ]:
class InContextMemory:
    """
    Manages the current conversation with sliding window pruning.
    Prevents context window overflow without losing recent turns.
    """
    def __init__(self, max_turns: int = 20):
        self.max_turns = max_turns
        self._history: list[dict] = []

    def add(self, role: str, content: Any) -> None:
        self._history.append({"role": role, "content": content})

    def get_window(self) -> list[dict]:
        # 2 messages per turn (user + assistant)
        return self._history[-(self.max_turns * 2):]

    def clear(self) -> None:
        self._history = []

    def __len__(self) -> int:
        return len(self._history)

ctx = InContextMemory(max_turns=3)
ctx.add("user", "Hi, I want to lose weight")
ctx.add("assistant", "Great! Let me help you with that.")
ctx.add("user", "What should I eat for breakfast?")
ctx.add("assistant", "I recommend eggs and oats.")
ctx.add("user", "Thanks! What about lunch?")
ctx.add("assistant", "For lunch, try chicken and vegetables.")
# Only keeps last 3 turns (6 messages)
print(f"History: {len(ctx)} messages | Window: {len(ctx.get_window())} messages")

## 5.5 · The MemoryEnabledDietCoach

In [ ]:
class MemoryEnabledDietCoach:
    """
    Section 5.5: Diet Coach with all three memory layers combined.
    
    Before each response:
      1. Retrieve relevant episodes (RAG pattern)
      2. Load semantic profile
      3. Compose context window
      4. Generate response
    
    After each session:
      5. Summarise and store as new episode
    """
    def __init__(self, user_id: str):
        self.user_id  = user_id
        self.semantic = SemanticMemory(user_id)
        self.episodic = EpisodicMemory(user_id)
        self.context  = InContextMemory(max_turns=10)
        self.client   = OpenAI()

    def _build_system_prompt(self, query: str) -> str:
        """Compose system prompt with all memory layers injected."""
        profile   = self.semantic.as_context_string()
        history   = self.episodic.as_context_string(query=query, n=2)
        return (
            "You are an AI Diet Coach.\n\n"
            f"[User Profile]\n{profile}\n\n"
            f"[Session History]\n{history}\n\n"
            "[Your Skill]\nConduct assessments: Baseline → Deficits → Priorities → One Goal.\n\n"
            "[Rules]\nNever diagnose. Refer medical concerns to a professional."
        )

    def chat(self, user_message: str) -> str:
        system = self._build_system_prompt(user_message)
        self.context.add("user", user_message)
        # System prompt prepended as first message — OpenAI convention
        response = self.client.chat.completions.create(
            model=MODEL, max_tokens=1024,
            messages=[
                {"role": "system", "content": system},
                *self.context.get_window(),
            ],
        )
        reply = response.choices[0].message.content
        self.context.add("assistant", reply)
        return reply

    def end_session(self) -> None:
        """Summarise the session and store as an episode."""
        if len(self.context) < 2:
            return
        transcript = "\n".join(
            f"{m['role']}: {str(m['content'])[:200]}"
            for m in self.context.get_window()
        )
        summary_prompt = (
            f"Summarise this coaching session in one sentence:\n{transcript}"
        )
        summary_resp = self.client.chat.completions.create(
            model=MODEL, max_tokens=100,
            messages=[{"role": "user", "content": summary_prompt}]
        )
        summary = summary_resp.choices[0].message.content.strip()
        self.episodic.add_episode(
            summary=summary,
            key_insights=[],
            goal_set=None,
        )
        print(f"Session stored: {summary[:80]}...")

# Demo
coach = MemoryEnabledDietCoach("alex_demo")
print(coach.chat("Hi, remind me — what was my goal from last time?"))
print("---")
print(coach.chat("I had oats and eggs for breakfast today. How am I doing?"))

## 5.6 · Context Engineering

Memory stores answer **what** the agent knows. Context engineering answers a different question: **what does the model see right now**, in what order, and at what level of detail?

Every token in the context window is a decision. The `build_context_window` function below assembles that window deliberately across three layers:

| Layer | Content | Why this order |
|-------|---------|----------------|
| 0 | Skill protocol | Injected via system prompt — highest attention weight |
| 1 | User profile | Stable background facts; changes rarely |
| 2 | Recent meals | Dynamic, recency-weighted; last 3 only |
| 3 | Current message | Always last — model weights recent tokens most heavily |

The **fake assistant acknowledgement turns** (`'Understood — I have your profile loaded.'`) are a deliberate priming technique. They condition the model to treat injected context as already-acknowledged information rather than summarising it back to the user.

In [ ]:
# Load the Skill protocol once at module level.
# build_context_window uses it as a default so callers don't need to
# pass it explicitly — consistent with how MemoryEnabledDietCoach loads it.
_skill_path = Path("../chapter_03_prompting/SKILL.md")
NUTRITION_ASSESSMENT_SKILL: str = (
    _skill_path.read_text() if _skill_path.exists() else ""
)


def build_context_window(
    user_message: str,
    meal_history: list[dict] | None = None,
    user_profile: dict | None = None,
    skill_text: str = NUTRITION_ASSESSMENT_SKILL,
) -> list[dict]:
    """
    Section 5.6: Assemble the message list the model will see.

    Treats every token as a deliberate decision rather than passing
    everything into the context and hoping the model pays attention
    to the right things.

    Args:
        user_message:  The user's current input — always the final message.
        meal_history:  Full meal log; only the last 3 entries are included.
        user_profile:  Dict of user facts (weight, goal, restrictions).
        skill_text:    The Nutrition Assessment Skill from SKILL.md.

    Returns:
        Message list in OpenAI API format, ordered by relevance.
    """
    messages: list[dict] = []

    # Layer 1 — User profile (stable background context)
    # Injected first; the synthetic assistant reply primes the model
    # to treat this as already-acknowledged rather than repeating it.
    if user_profile:
        profile_note = f"[Context] User profile:\n{json.dumps(user_profile, indent=2)}"
        messages.append({"role": "user",      "content": profile_note})
        messages.append({"role": "assistant", "content": "Understood — I have your profile loaded."})

    # Layer 2 — Recent meals (dynamic, recency-weighted)
    # Limit to the last 3: older entries are noise. The 'lost in the
    # middle' problem means information buried deep in long contexts
    # receives less model attention.
    if meal_history:
        recent = meal_history[-3:]
        history_note = (
            "Recent meals logged:\n"
            + "\n".join(
                f"  - {m.get('food', '?')} ({m.get('meal_type', '?')})"
                for m in recent
            )
        )
        messages.append({"role": "user",      "content": history_note})
        messages.append({"role": "assistant", "content": "Got it — I'll factor in your recent meals."})

    # Layer 3 — Current message (always last, always highest weight)
    messages.append({"role": "user", "content": user_message})
    return messages


print("build_context_window defined.")

### Demonstration — what the model actually sees

Run the cell below to inspect the assembled context window. Notice:
- The apple snack is **excluded** — it falls outside the last-3 window
- The profile and meal history arrive as **fake dialogue turns**, not raw JSON injected into the system prompt
- The user's actual question is always the **final message**

In [ ]:
user_profile = {
    "name":           "Jordan",
    "age":            32,
    "weight_kg":      78,
    "goal":           "Lose 5 kg over 3 months",
    "restrictions":   "Lactose intolerant",
    "activity_level": "Runs 3x per week",
}

meal_history = [
    {"food": "oats with banana",       "meal_type": "breakfast"},
    {"food": "chicken salad",          "meal_type": "lunch"},
    {"food": "salmon with brown rice", "meal_type": "dinner"},
    {"food": "apple",                  "meal_type": "snack"},   # dropped — outside last 3
    {"food": "eggs and smoked salmon", "meal_type": "breakfast"},
]

messages = build_context_window(
    user_message="How am I doing on protein today?",
    meal_history=meal_history,
    user_profile=user_profile,
)

print(f"Context window: {len(messages)} messages\n")
for i, msg in enumerate(messages):
    preview = msg["content"][:80] + "..." if len(msg["content"]) > 80 else msg["content"]
    print(f"  [{i}] {msg['role']:10s} | {preview}")

print("\nLayer breakdown:")
print("  Messages 0–1 : Layer 1 — user profile (stable background)")
print("  Messages 2–3 : Layer 2 — last 3 meals (chicken salad, salmon, eggs)")
print("  Message  4   : Layer 3 — current question (highest attention weight)")
print("\nNote: 'apple' excluded — falls outside the last-3 window.")

### How context engineering connects to memory

`build_context_window` and `MemoryEnabledDietCoach` address different parts of the same problem:

- **MemoryEnabledDietCoach** decides *what to store and retrieve* — writing to semantic memory after profile updates, retrieving relevant episodes via RAG, maintaining the sliding in-context window.
- **build_context_window** decides *how to present what was retrieved* — ordering layers by relevance, limiting recency to last-3 meals, using synthetic acknowledgement turns to prime the model.

In a production system you would call `build_context_window` inside `MemoryEnabledDietCoach.chat()`, replacing the raw `self.in_context.get_window()` call with a deliberately engineered context assembly. The memory stores provide the raw material; context engineering determines how that material is shaped into the model's view of the world.

## 5.7 · Chapter Summary

| Memory type | What we built | Production path |
|---|---|---|
| Semantic | JSON key-value store | Redis / DynamoDB |
| Episodic | Session summaries + keyword retrieval | Vector DB + embeddings |
| In-context | Sliding window with `max_turns` | Same — already optimal |
| Procedural | SKILL.md (Chapter 3) | Skill registry + semantic search |
| Context | `build_context_window` — layered assembly | Same pattern; swap to streaming context with vector retrieval |

**What is next?** Chapter 6 — Communication: the protocols that let multiple agents talk to each other reliably.

---
*Mastering Agentic AI · Manning Publications*